# Data Prep: EV Charging Patterns (Dataset 4)

Raw file: `data/raw/4-ev_charging_patterns.csv`

**Target (regression):** `Energy Consumed (kWh)`

Pipeline aligned with `5-EV_energy_consumption_data_prep.ipynb`: missing values → duplicates → categorical encoding → outlier capping → 60/20/20 split → `StandardScaler` on continuous features (train-fit only) → export to `data/processed/`.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_PATH = "data/raw/4-ev_charging_patterns.csv"
TARGET_COL = "Energy Consumed (kWh)"

data = pd.read_csv(DATA_PATH)
print("Shape:", data.shape)
data.head()


## Step 1: Initial Data Exploration


In [ ]:
print("--- Data types ---")
print(data.dtypes)
print("\n--- Missing values ---")
print(data.isna().sum().sort_values(ascending=False))
print("\n--- Basic stats (numeric) ---")
print(data.describe(include=[np.number]))


## Step 2: Handling Missing Values


In [ ]:
# Drop columns that are mostly missing (if any)
threshold = len(data) * 0.5
data_clean = data.dropna(thresh=threshold, axis=1)

# Drop identifiers / raw timestamps (not used as features in this prep)
drop_id_cols = [
    "User ID",
    "Charging Station ID",
    "Charging Start Time",
    "Charging End Time",
]
data_clean = data_clean.drop(columns=drop_id_cols, errors="ignore")

# Numeric: median imputation
numeric_cols = data_clean.select_dtypes(include=np.number).columns
data_clean[numeric_cols] = data_clean[numeric_cols].fillna(data_clean[numeric_cols].median())

# Categorical: mode imputation
categorical_cols = data_clean.select_dtypes(include="object").columns
for col in categorical_cols:
    mode_series = data_clean[col].mode(dropna=True)
    fill = mode_series.iloc[0] if len(mode_series) else "unknown"
    data_clean[col] = data_clean[col].fillna(fill)

print("Missing after imputation:", data_clean.isna().sum().sum())
print("Shape:", data_clean.shape)


## Step 3: Dealing with Duplicates


In [ ]:
print("Duplicate rows:", data_clean.duplicated().sum())
data_clean = data_clean.drop_duplicates()
print("Shape after drop_duplicates:", data_clean.shape)


## Step 4: Handling Categorical Data


In [ ]:
# Normalize text
for col in data_clean.select_dtypes(include="object").columns:
    data_clean[col] = data_clean[col].astype(str).str.strip().str.lower()

nominal_cols = [
    "Vehicle Model",
    "Charging Station Location",
    "Time of Day",
    "Day of Week",
    "Charger Type",
    "User Type",
]
nominal_cols = [c for c in nominal_cols if c in data_clean.columns]

data_encoded = pd.get_dummies(data_clean, columns=nominal_cols, drop_first=False)
print("Shape after one-hot:", data_encoded.shape)
data_encoded.head()


## Step 5: Outlier Management (±3σ cap on continuous numeric columns)


In [ ]:
numeric_features = data_encoded.select_dtypes(include=np.number).columns.tolist()
# Skip target and binary dummy columns from capping
exclude_cols = [TARGET_COL] + [c for c in numeric_features if data_encoded[c].nunique() <= 2]
cap_cols = [c for c in numeric_features if c not in exclude_cols]

outlier_counts = {}
for col in cap_cols:
    mean, std = data_encoded[col].mean(), data_encoded[col].std()
    if std == 0 or pd.isna(std):
        continue
    lower, upper = mean - 3 * std, mean + 3 * std
    n_out = ((data_encoded[col] < lower) | (data_encoded[col] > upper)).sum()
    if n_out > 0:
        outlier_counts[col] = int(n_out)
    data_encoded[col] = data_encoded[col].clip(lower, upper)

print("Outliers capped:", outlier_counts)
print("Shape:", data_encoded.shape)


## Step 6: Data Partitioning (60% / 20% / 20%)


In [ ]:
df_model = data_encoded.copy()
X = df_model.drop(columns=[TARGET_COL])
y = df_model[TARGET_COL]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

print(f"Training set:    {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set:  {X_val.shape[0]} rows ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set:        {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.1f}%)")
print("\nTarget (train) stats:")
print(y_train.describe())


## Step 7: Feature Scaling (StandardScaler, train-fit only)


In [ ]:
scale_cols = [
    c for c in X_train.select_dtypes(include=np.number).columns
    if X_train[c].nunique() > 2
]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val_scaled[scale_cols] = scaler.transform(X_val[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

print("Standardized columns:", scale_cols)

minmax = MinMaxScaler()
X_train_norm = X_train.copy()
X_val_norm = X_val.copy()
X_test_norm = X_test.copy()
X_train_norm[scale_cols] = minmax.fit_transform(X_train[scale_cols])
X_val_norm[scale_cols] = minmax.transform(X_val[scale_cols])
X_test_norm[scale_cols] = minmax.transform(X_test[scale_cols])


## Step 8: Visual Exploration


In [ ]:
# Univariate: key numerics from raw-named columns (pre-dummy)
plot_cols = [
    c for c in [
        "Battery Capacity (kWh)",
        "Charging Duration (hours)",
        "Charging Rate (kW)",
        "Charging Cost (USD)",
        "State of Charge (Start %)",
        "State of Charge (End %)",
        "Distance Driven (since last charge) (km)",
        TARGET_COL,
    ]
    if c in data_clean.columns
]

n = len(plot_cols)
fig, axes = plt.subplots(int(np.ceil(n / 3)), 3, figsize=(14, 3.5 * int(np.ceil(n / 3))))
axes = np.array(axes).reshape(-1)
for i, col in enumerate(plot_cols):
    sns.histplot(data_clean[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(col)
for j in range(len(plot_cols), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# Correlation among numeric columns in cleaned (pre-dummy) data
num_only = data_clean.select_dtypes(include=[np.number])
plt.figure(figsize=(12, 9))
sns.heatmap(num_only.corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation heatmap (numeric features, pre one-hot)")
plt.tight_layout()
plt.show()


## Export: Processed Dataset + Splits

| Output | Description |
|--------|-------------|
| `4-ev_charging_patterns_processed.csv` | Full encoded + capped frame |
| `4-ev_charging_patterns_X_{train,val,test}.csv` | Standardized **X** |
| `4-ev_charging_patterns_y_{train,val,test}.csv` | Target **y** |

**Use case:** Compare driving / charging behaviors (session-level features vs energy delivered).


In [ ]:
os.makedirs("data/processed", exist_ok=True)

full_processed = data_encoded.copy()
full_processed.to_csv("data/processed/4-ev_charging_patterns_processed.csv", index=False)
print("Saved:", full_processed.shape, "→ data/processed/4-ev_charging_patterns_processed.csv")

X_train_scaled.to_csv("data/processed/4-ev_charging_patterns_X_train.csv", index=False)
X_val_scaled.to_csv("data/processed/4-ev_charging_patterns_X_val.csv", index=False)
X_test_scaled.to_csv("data/processed/4-ev_charging_patterns_X_test.csv", index=False)
y_train.to_csv("data/processed/4-ev_charging_patterns_y_train.csv", index=False, header=True)
y_val.to_csv("data/processed/4-ev_charging_patterns_y_val.csv", index=False, header=True)
y_test.to_csv("data/processed/4-ev_charging_patterns_y_test.csv", index=False, header=True)

print("Saved train/val/test splits (StandardScaler on continuous columns).")
print("Target column:", TARGET_COL)
